# NVIDIA Nemotron 3.5 Super VL — new customer-service agent with LangChain

This is the **LangChain version** of the Northline Audio return-case demo. It creates a **separate Foundry agent** named `nemotron-customer-service-langchain`, attaches the existing Northline file-search vector store, and runs the same synthetic order case. It does not modify the earlier `nemotron-customer-service-returns` agent.

| Step | Component | What it does |
| --- | --- | --- |
| See | `AzureAIOpenAIApiChatModel` | Sends the order image to the Nemotron deployment through Foundry **Chat Completions**. |
| Act | `AgentServiceFactory.create_prompt_agent` | Creates and runs a new managed Foundry agent with a local `read_order_document` function. |
| Know | Foundry file search | Retrieves the Northline returns policy and Aurora H7 support bulletin from the existing vector store. |
| Observe | Foundry conversation and traces | Keeps the new agent and conversation for inspection in the portal. |

The notebook calls **LangChain APIs for model inference and agent orchestration**. `azure-identity` supplies Microsoft Entra credentials. The declarative `FileSearchTool` type comes from the Azure package because that is how `langchain-azure-ai` attaches Foundry's server-side file search; there are no direct `AIProjectClient` or OpenAI SDK calls in the notebook. The image stays in the local function path because this deployment's agent Responses endpoint did not accept direct image input in the prior demo.

[LangChain + Foundry agents](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/langchain-agents) · [LangChain + Foundry models](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/langchain-models) · [Nemotron article](https://developer.nvidia.com/blog/nvidia-nemotron-3-5-super-vl-powers-specialized-agents-with-multimodal-reasoning/)

## Setup

Follow [the sample README](README.md) to set up this notebook's Python environment and Microsoft Entra authentication. Run the direct Foundry SDK notebook once to index the two synthetic knowledge documents, then export its vector store ID. Run cells in order. The notebook does not delete its agent or conversations.


In [ ]:
import base64
import os
from pathlib import Path

from IPython.display import Image, Markdown, display
from azure.identity import DefaultAzureCredential
from langchain_core.messages import HumanMessage, ToolMessage
from azure.ai.projects.models import FileSearchTool
from langchain_core.tools import tool
from langchain_azure_ai.agents import AgentServiceFactory
from langchain_azure_ai.agents.prebuilt.tools import AgentServiceBaseTool
from langchain_azure_ai.chat_models import AzureAIOpenAIApiChatModel
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode

In [ ]:
PROJECT_ENDPOINT = os.environ.get("FOUNDRY_PROJECT_ENDPOINT", "").rstrip("/")
MODEL = os.environ.get("FOUNDRY_MODEL", "")
if not PROJECT_ENDPOINT or not MODEL:
    raise EnvironmentError("Set FOUNDRY_PROJECT_ENDPOINT and FOUNDRY_MODEL before running this notebook.")
AGENT_NAME = os.getenv("FOUNDRY_LANGCHAIN_AGENT_NAME", "nemotron-customer-service-langchain")
VECTOR_STORE_ID = os.environ.get("FOUNDRY_CUSTOMER_SERVICE_VECTOR_STORE_ID", "")
if not VECTOR_STORE_ID:
    raise EnvironmentError("Set FOUNDRY_CUSTOMER_SERVICE_VECTOR_STORE_ID from the direct SDK notebook output.")
ASSET_DIR = next(
    (p for p in [
        Path.cwd() / "customer_service_assets",
        Path.cwd() / "inference" / "msfoundry" / "nemotron-35-vl-super" / "customer_service_assets",
    ] if (p / "northline_order_return_case.png").is_file()),
    None,
)
if ASSET_DIR is None:
    raise FileNotFoundError("Open the notebook from its directory or the repository root.")
IMAGE_PATH = ASSET_DIR / "northline_order_return_case.png"
for filename in ("northline_returns_policy.txt", "aurora_h7_support_bulletin.txt"):
    if not (ASSET_DIR / filename).is_file():
        raise FileNotFoundError(ASSET_DIR / filename)
credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
credential.get_token("https://ai.azure.com/.default")
print("Microsoft Entra sign-in ready")
print(f"New agent: {AGENT_NAME}")
print(f"Existing knowledge vector store: {VECTOR_STORE_ID}")
print(f"Model: {MODEL}")

## 1. Customer image

The order image contains the item table, shipment-event table, and the customer's return request. The complaint and attempted reset are **customer statements**, not verified defects.

In [ ]:
display(Image(filename=str(IMAGE_PATH), width=950))

## 2. LangChain vision function

`AzureAIOpenAIApiChatModel` calls the deployed Nemotron model using **Chat Completions** and Microsoft Entra authentication. The tool returns only visual facts. Return eligibility is decided later using the Foundry knowledge sources.

In [ ]:
vision_model = AzureAIOpenAIApiChatModel(
    project_endpoint=PROJECT_ENDPOINT,
    credential=credential,
    model=MODEL,
    use_responses_api=False,
    timeout=120,
)

@tool
def read_order_document() -> str:
    """Read the customer-supplied order and return-case image; report only visible facts."""
    image_url = "data:image/png;base64," + base64.b64encode(IMAGE_PATH.read_bytes()).decode("ascii")
    message = HumanMessage(content=[
        {"type": "text", "text": (
            "Read the attached order and return-case document. Transcribe the order ID, "
            "product, SKU, price, order date, delivery date from the shipment table, "
            "case review date, return-request date, reported issue, reported troubleshooting, "
            "and listed box contents. Distinguish recorded order/shipment facts from "
            "the customer's unverified statements. Do not apply any return policy. "
            "Mark any unreadable field as unknown."
        )},
        {"type": "image_url", "image_url": {"url": image_url}},
    ])
    result = vision_model.invoke([message])
    facts = result.content
    if not isinstance(facts, str) or not facts.strip():
        raise RuntimeError(f"Nemotron returned no text facts: {facts!r}")
    return facts

## 3. Create a dedicated Foundry agent with LangChain

`create_prompt_agent` creates a **new managed agent** using NVIDIA Nemotron 3.5 Super VL. LangGraph runs `read_order_document` locally. Foundry runs file search against the existing vector store and keeps the conversation and server-side spans. The policy and support bulletin are already indexed, so this cell does not upload them again.

In [ ]:
factory = AgentServiceFactory(project_endpoint=PROJECT_ENDPOINT, credential=credential)
file_search = AgentServiceBaseTool(tool=FileSearchTool(vector_store_ids=[VECTOR_STORE_ID]))
INSTRUCTIONS = (
    "You are Northline Audio's customer-service returns specialist in a fictional case. "
    "For order, shipment, item, and customer-statement facts, call read_order_document "
    "unless those visual facts are already in the conversation. For policy, dates, "
    "shipping, fees, refund timing, troubleshooting, or escalation, use file search "
    "to consult both the Northline Audio Returns Policy and Aurora H7 No-Audio Support Bulletin. "
    "The case review date is 2026-10-21. Count calendar days from the day after delivery. "
    "Separate visual facts, unverified customer statements, policy rules, and support guidance. "
    "Name the source title in plain text after each nonvisual claim. "
    "Do not say a defect is confirmed, an RMA is issued, or a refund is approved. "
    "Only explain conditional eligibility and the steps a human support team must complete."
)
service_graph = factory.create_prompt_agent(
    name=AGENT_NAME,
    model=MODEL,
    description="Northline Audio returns specialist using Nemotron vision and Foundry policy search",
    instructions=INSTRUCTIONS,
    tools=[read_order_document, file_search],
    checkpointer=InMemorySaver(),
)
agent_ids = factory.get_agents_id_from_graph(service_graph)
print("Created Foundry agent version:", ", ".join(sorted(agent_ids)))
print("Open https://ai.azure.com/ to inspect the new agent.")
THREAD_ID = "northline-customer-service-langchain-live-demo"
RUN_CONFIG = {"configurable": {"thread_id": THREAD_ID}}

## 4. Question runner

Each question is a separate notebook cell. The same LangGraph thread carries the new agent's Foundry conversation forward. Tool outputs are printed for the live demo; no agent output or conversation is deleted.

In [ ]:
def ask_customer_agent(question: str, *, expect_visual_tool: bool = False):
    result = service_graph.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={**RUN_CONFIG, "recursion_limit": 12},
    )
    conversation_id = result.get("azure_ai_agents_conversation_id")
    if not conversation_id:
        raise RuntimeError("Foundry returned no conversation ID")
    tool_messages = [m for m in result["messages"] if isinstance(m, ToolMessage)]
    if expect_visual_tool and not any(m.name == "read_order_document" for m in tool_messages):
        raise RuntimeError("The agent did not request read_order_document")
    answer = result["messages"][-1]
    if getattr(answer, "tool_calls", None):
        raise RuntimeError("The agent still has an unresolved tool call")
    print(f"Foundry conversation: {conversation_id}")
    print(f"Local function results in thread: {len(tool_messages)}")
    print("Open https://ai.azure.com/ and search for this conversation in the agent Traces view.")
    display(Markdown(str(answer.content)))
    return result


def show_visual_tool_output(result):
    for message in result["messages"]:
        if isinstance(message, ToolMessage) and message.name == "read_order_document":
            display(Markdown("### `read_order_document` output — Nemotron visual facts"))
            display(Markdown(str(message.content)))
            return
    print("No visual function output in this thread yet.")

## 5. Question 1 — read the visual document

The agent asks for `read_order_document`; the LangGraph tool node then calls Nemotron on the image.

In [ ]:
case_1 = ask_customer_agent(
    "Read the attached Northline order and return-case document. What was ordered, "
    "when was it delivered, when was the return request submitted, and what does the "
    "customer say happened? Call read_order_document and separate recorded facts "
    "from unverified customer statements.",
    expect_visual_tool=True,
)
show_visual_tool_output(case_1)

## 6. Question 2 — search company knowledge

The Foundry agent's existing file-search tool retrieves the **Northline Audio Returns Policy** and **Aurora H7 No-Audio Support Bulletin** from its vector store. Python computes the day counts deterministically, then asks the agent to apply the retrieved rules and name each source in plain text. Inspect the portal trace to confirm the file-search step. This notebook does not substitute a local text-file lookup for Foundry knowledge.

In [ ]:
from datetime import date, timedelta

delivery_date = date(2026, 9, 18)
review_date = date(2026, 10, 21)
standard_deadline = delivery_date + timedelta(days=30)
defect_review_deadline = delivery_date + timedelta(days=45)
print(
    f"Calendar check: {(review_date - delivery_date).days} days since delivery; "
    f"standard deadline {standard_deadline} ({(review_date - standard_deadline).days} days ago); "
    f"defect-review deadline {defect_review_deadline} "
    f"({(defect_review_deadline - review_date).days} days away)."
)
case_2 = ask_customer_agent(
    f"As of {review_date}, is this case within the standard-return window or the "
    f"suspected-defect review window? The recorded delivery date is {delivery_date}; "
    f"33 calendar days have elapsed. The standard deadline was {standard_deadline}, "
    f"3 days ago. The defect-review deadline is {defect_review_deadline}, "
    f"12 days away. Use these exact dates and differences, then explain shipping "
    "and refund conditions. Search both the Northline Audio Returns Policy and "
    "Aurora H7 No-Audio Support Bulletin in Foundry file search. Separate "
    "policy rules from troubleshooting guidance and name both source titles."
)

## 7. Question 3 — customer response

Draft a conditional reply. The agent must not call the customer's reported issue a confirmed hardware defect or promise an RMA or immediate refund.

In [ ]:
case_3 = ask_customer_agent(
    "Draft a concise customer-facing reply. Explain the remaining balance, "
    "firmware, and second-device checks, what support should do if the issue "
    "persists, and the conditional return path. Cite the returns policy and "
    "support bulletin by title. Do not say the RMA or refund is approved."
)

## Expected interpretation and trace notes

- The recorded delivery date is **2026-09-18**. On **2026-10-21**, **33 calendar days** have elapsed. The 30-day standard window has passed, while the 45-day suspected-defect review ends **2026-11-02**.
- The image reports a right-earcup problem and a reset attempt; support has not verified the defect or all box contents.
- The policy supplies return windows, shipping, fees, and refund timing. The bulletin supplies charge/balance, firmware, and second-device troubleshooting.
- Foundry runs file search against the reused vector store. The new agent and conversation remain in the project. The LangGraph state exposes the conversation ID; the Foundry **Traces** page shows server-side execution if Application Insights is enabled and ingestion has completed. Search the printed conversation ID there.

If the agent does not use file search or cites neither source, rerun the knowledge question with an explicit request to search both documents and inspect the trace. Do not present an unsupported answer as policy-grounded.